## Data Loading and Packages

In [2]:
### Load libraries
import pandas as pd
import numpy as np

## Load data 
cols_used = ["player_name",
    "pitcher",
    "pitch_type",
    "release_speed",
    "release_spin_rate",
    "spin_axis",
    "release_pos_x",
    "release_pos_y",
    "release_pos_z",
    "release_extension",
    "vx0",
    "vy0",
    "vz0",
    "ax",
    "ay",
    "az"]

data = pd.read_csv(r'C:\Users\choul\OneDrive\Baseball Repo\Baseball-Analytics\data\MLB_2021-2025.csv')[cols_used].dropna(subset = cols_used)
data.head()

,player_name,pitcher,pitch_type,release_speed,release_spin_rate,spin_axis,release_pos_x,release_pos_y,release_pos_z,release_extension,vx0,vy0,vz0,ax,ay,az
0,"Smith, Will",519293,FF,92.3,2330.0,148.0,1.40,54.03,6.80,6.5,-6.833043,-134.166485,-7.361843,9.708393,26.562803,-14.083224
1,"Smith, Will",519293,SL,80.6,2254.0,315.0,1.60,54.15,6.64,6.4,-3.700232,-117.430885,-3.266842,-6.531123,19.793390,-27.369114
2,"Smith, Will",519293,CU,75.5,1940.0,328.0,1.46,54.34,6.88,6.2,-1.977183,-109.901781,-1.155694,-4.872924,20.602334,-36.262184
3,"Smith, Will",519293,CU,75.0,2017.0,330.0,1.53,54.61,6.83,5.9,2.375830,-109.205830,2.277617,-5.902656,19.427562,-38.284747
4,"Smith, Will",519293,FF,91.2,2281.0,143.0,1.49,54.15,6.66,6.3,-5.868477,-132.500539,-6.486796,8.700586,30.117690,-15.941174


## Feature Engineering

#### Trajectory Calculation

In [ ]:
df = data.copy()
### Ball position @ Y feet from plate
TRAJ_DISTANCES = [40, 30, 20, 10]

### Adjust spin axis to radians, calculate sin/cos
theta = np.deg2rad(df['spin_axis'])
df['spin_axis_sin'] = np.sin(theta)
df['spin_axis_cos'] = np.cos(theta) 

### Derived physics
df['velocity_mag'] = np.sqrt(df['vx0']**2, df["vy0"]**2 + df["vz0"]**2)
df["acceleration_mag"] = np.sqrt( df["ax"]**2 + df["ay"]**2 + df["az"]**2 )
df["horizontal_velocity"] = np.sqrt( df["vx0"]**2 + df["vy0"]**2 ) 
df["horizontal_acceleration"] = np.sqrt( df["ax"]**2 + df["ay"]**2 )

### 
def solve_time_to_y(y0, vy0, ay, target_y):
    """
    Solve:

        target_y = y0 + vy0*t + 0.5*ay*t^2

    for the positive time at which the pitch reaches target_y.
    """

    a = 0.5 * ay
    b = vy0
    c = y0 - target_y

    discriminant = b**2 - 4 * a * c

    if discriminant < 0:
        return np.nan

    # Approximately linear motion
    if abs(a) < 1e-10:
        if abs(b) < 1e-10:
            return np.nan

        t = -c / b
        return t if t > 0 else np.nan

    sqrt_disc = np.sqrt(discriminant)

    t1 = (-b + sqrt_disc) / (2 * a)
    t2 = (-b - sqrt_disc) / (2 * a)

    valid_times = [
        t for t in (t1, t2)
        if t > 0
    ]

    if not valid_times:
        return np.nan

    return min(valid_times)


def trajectory_at_distance(row, distance):
    """
    Calculate pitch position and velocity at a specified
    distance from the release point toward home plate.
    """

    target_y = row["release_pos_y"] - distance

    t = solve_time_to_y(
        row["release_pos_y"],
        row["vy0"],
        row["ay"],
        target_y
    )

    if pd.isna(t):
        return pd.Series({
            f"x_{distance}ft": np.nan,
            f"z_{distance}ft": np.nan,
            f"vx_{distance}ft": np.nan,
            f"vy_{distance}ft": np.nan,
            f"vz_{distance}ft": np.nan,
            f"speed_{distance}ft": np.nan,
            f"time_{distance}ft": np.nan,
        })

    x = (
        row["release_pos_x"]
        + row["vx0"] * t
        + 0.5 * row["ax"] * t**2
    )

    z = (
        row["release_pos_z"]
        + row["vz0"] * t
        + 0.5 * row["az"] * t**2
    )

    vx = row["vx0"] + row["ax"] * t
    vy = row["vy0"] + row["ay"] * t
    vz = row["vz0"] + row["az"] * t

    speed = np.sqrt(
        vx**2 +
        vy**2 +
        vz**2
    )

    return pd.Series({
        f"x_{distance}ft": x,
        f"z_{distance}ft": z,
        f"vx_{distance}ft": vx,
        f"vy_{distance}ft": vy,
        f"vz_{distance}ft": vz,
        f"speed_{distance}ft": speed,
        f"time_{distance}ft": t,
    })


# ============================================================
# CREATE V1 TRAJECTORY FEATURES
# ============================================================

for distance in TRAJ_DISTANCES:

    trajectory_features = df.apply(
        trajectory_at_distance,
        axis=1,
        distance=distance
    )

    df = pd.concat(
        [df, trajectory_features],
        axis=1
    )


# ============================================================
# ADDITIONAL TRAJECTORY FEATURES
# ============================================================

for distance in TRAJ_DISTANCES:

    df[f"horizontal_velocity_{distance}ft"] = np.sqrt(
        df[f"vx_{distance}ft"]**2 +
        df[f"vy_{distance}ft"]**2
    )

: 